In [3]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow import keras
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.metrics import classification_report
import datetime
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten
from sklearn.decomposition import PCA

I0000 00:00:1778522752.378998   84050 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778522752.420312   84050 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778522755.183186   84050 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Rúbrica: 

A continuación se muestra la rúbrica con la que se va a corregir el examen:


| Apartado/Criterio | Ponderación | 
| :-- | --- | 
| Ej. 1. Ha tratado de manera adecuada los datos de las columnas. | 0,5 | 
| Ej. 1. Ha usado como entrada las columnas adecuadas. | 0,5 |
| Ej. 1. Ha diseñado bien la red, el sistema de entrenamiento y ha comprobado si el resultado es bueno | 0,5 | 
| Ej. 1. Ha dimensionado bien la red neuronal. | 0,5 | 
| Ej. 1. Ha usado adecuadamente las técnicas de deep learning. | 2 | 
| Ej. 1. Ha tratado de realizar una predicción de forma adecuada. | 1,5 | 
| Ej. 2. Ha cargado los datos probando los dos métodos propuestos. | 1,5 | 
| Ej. 2. Ha diseñado bien la red convolucional y el sistema de entrenamiento | 1,5 | 
| Ej. 2. Ha dimensionado bien la red neuronal. | 0,5 | 
| Ej. 2. Ha usado las técnicas y el proceso visto en clase. | 0,5 | 
| Ej. 2. Ha usado su experiencia para valorar si el resultado es válido o no. | 0,5 | 



# EJERCICIO 1

In [86]:
df_destruction = pd.read_csv("destruccion_empleo.csv")
df_destruction.head()

,record_id,country,iso3_code,region,income_group,year,quarter,quarter_label,industry_sector,sector_automation_risk_score,gdp_per_capita_usd,pct_sector_workforce_displaced,data_source_notes
0,1,United States,USA,North America,High Income,2020,1,2020-Q1,Technology & Software,0.382,63514,0.0406,Research-calibrated synthetic data. Grounded i...
1,2,United States,USA,North America,High Income,2020,1,2020-Q1,Finance & Banking,0.608,63514,0.0517,Research-calibrated synthetic data. Grounded i...
2,3,United States,USA,North America,High Income,2020,1,2020-Q1,Healthcare & Life Sciences,0.198,63514,0.0176,Research-calibrated synthetic data. Grounded i...
3,4,United States,USA,North America,High Income,2020,1,2020-Q1,Manufacturing & Industry,0.720,63514,0.0924,Research-calibrated synthetic data. Grounded i...
4,5,United States,USA,North America,High Income,2020,1,2020-Q1,Retail & E-Commerce,0.676,63514,0.0667,Research-calibrated synthetic data. Grounded i...


In [87]:
df_destruction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20800 entries, 0 to 20799
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   record_id                       20800 non-null  int64  
 1   country                         20800 non-null  object 
 2   iso3_code                       20800 non-null  object 
 3   region                          20800 non-null  object 
 4   income_group                    20800 non-null  object 
 5   year                            20800 non-null  int64  
 6   quarter                         20800 non-null  int64  
 7   quarter_label                   20800 non-null  object 
 8   industry_sector                 20800 non-null  object 
 9   sector_automation_risk_score    20800 non-null  float64
 10  gdp_per_capita_usd              20800 non-null  int64  
 11  pct_sector_workforce_displaced  20800 non-null  float64
 12  data_source_notes               

In [88]:
df_destruction["industry_sector"].unique()

array(['Technology & Software', 'Finance & Banking',
       'Healthcare & Life Sciences', 'Manufacturing & Industry',
       'Retail & E-Commerce', 'Education & Research',
       'Transportation & Logistics', 'Media & Communications',
       'Administrative & Clerical', 'Energy & Utilities'], dtype=object)

In [89]:
#Borro quarter label porque teniendo ya el año y el cuatrimestre es tonteria tenerla, el iso3_code porque teniendo ya el pais también es innecesario ya que es lo mismo
#data_source_notes lo borro porque solo tiene un valor, se podría quitar region por lo mismo, tenemos los paises entonces se puede presuponer la region,
#pero la voy a dejar porque no estoy 100% seguro, record_id se borra porque es variable con valores diferentes todo el rato 
df_destruction.drop(columns=["quarter_label", "iso3_code","data_source_notes", "record_id"], inplace=True)

In [90]:
#No hay nulos asi que esa parte de limpieza nos la quitamos, por lo que realizo un dummies para la limpieza de variables object
df_destruction = pd.get_dummies(df_destruction,dtype=int)

In [91]:
print(df_destruction.shape)
df_destruction.head()

(20800, 111)


,year,quarter,sector_automation_risk_score,gdp_per_capita_usd,pct_sector_workforce_displaced,country_Algeria,country_Argentina,country_Australia,country_Austria,country_Bangladesh,...,industry_sector_Administrative & Clerical,industry_sector_Education & Research,industry_sector_Energy & Utilities,industry_sector_Finance & Banking,industry_sector_Healthcare & Life Sciences,industry_sector_Manufacturing & Industry,industry_sector_Media & Communications,industry_sector_Retail & E-Commerce,industry_sector_Technology & Software,industry_sector_Transportation & Logistics
0,2020,1,0.382,63514,0.0406,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,2020,1,0.608,63514,0.0517,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,2020,1,0.198,63514,0.0176,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,2020,1,0.720,63514,0.0924,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,2020,1,0.676,63514,0.0667,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [92]:
X = df_destruction[0:5000].drop(columns=["pct_sector_workforce_displaced"])
y = df_destruction["pct_sector_workforce_displaced"][0:5000]

In [93]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [94]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [95]:
#Tiro ramdon forest regressor por si acaso
rf = RandomForestRegressor(n_estimators=700, n_jobs=-1)
rf.fit(X_train,y_train)
print("R² score", rf.score(X_test,y_test))

R² score 0.9465143908430713


In [ ]:
model = keras.models.Sequential()
model.add(keras.layers.Dense(512,input_shape=X_train.shape[1:], activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(256,activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1, kernel_initializer='glorot_normal'))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [110]:
model = keras.models.Sequential()
model.add(keras.layers.Dense(2048,input_shape=X_train.shape[1:], activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(512,activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1, kernel_initializer='glorot_normal'))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [103]:
model = keras.models.Sequential()
model.add(keras.layers.Dense(3000,input_shape=X_train.shape[1:], activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(800,activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1, kernel_initializer='glorot_normal'))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [111]:
model = keras.models.Sequential()
model.add(keras.layers.Dense(2000,input_shape=X_train.shape[1:], activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(500,activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1, kernel_initializer='glorot_normal'))

In [114]:
model = keras.models.Sequential()
model.add(keras.layers.Dense(2048,input_shape=X_train.shape[1:], activation='relu', kernel_initializer='he_normal'))
# model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(512,activation='relu', kernel_initializer='he_normal'))
# model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1, kernel_initializer='glorot_normal'))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [115]:
#He tenido que reducir la cantidad de fotos porque me tarda 40 segundos por epoca
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
model.compile(loss='mean_absolute_error', metrics=['mae'],optimizer = keras.optimizers.Adam(learning_rate=0.0001,beta_1=0.9,beta_2=0.999))
history = model.fit(X_train, y_train, epochs=100000, validation_split = 0.1, callbacks=[early_stopping_cb])


Epoch 1/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.2113 - mae: 0.2113 - val_loss: 0.1472 - val_mae: 0.1472
Epoch 2/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1394 - mae: 0.1394 - val_loss: 0.1315 - val_mae: 0.1315
Epoch 3/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1308 - mae: 0.1308 - val_loss: 0.1058 - val_mae: 0.1058
Epoch 4/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1079 - mae: 0.1079 - val_loss: 0.1016 - val_mae: 0.1016
Epoch 5/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1022 - mae: 0.1022 - val_loss: 0.0935 - val_mae: 0.0935
Epoch 6/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0968 - mae: 0.0968 - val_loss: 0.1107 - val_mae: 0.1107
Epoch 7/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0865 - mae: 0.0865 - val_loss: 0.1017 - val_mae: 0.1017
Epoch 8/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0857 - mae: 0.0857 - val_loss: 0.0778 - val_mae: 0.0778
Epoch 9/100000
113/113 ━

In [108]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
model.compile(loss='mean_absolute_error', metrics=['mae'],optimizer = keras.optimizers.RMSprop(learning_rate=0.0001,rho=0.9))
history = model.fit(X_train, y_train, epochs=100000 , validation_split = 0.1, callbacks=[early_stopping_cb])

Epoch 1/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.5652 - mae: 0.5652 - val_loss: 0.4449 - val_mae: 0.4449
Epoch 2/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.4764 - mae: 0.4764 - val_loss: 0.4496 - val_mae: 0.4496
Epoch 3/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.4189 - mae: 0.4189 - val_loss: 0.3891 - val_mae: 0.3891
Epoch 4/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.3930 - mae: 0.3930 - val_loss: 0.3623 - val_mae: 0.3623
Epoch 5/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.3662 - mae: 0.3662 - val_loss: 0.4243 - val_mae: 0.4243
Epoch 6/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.3392 - mae: 0.3392 - val_loss: 0.3811 - val_mae: 0.3811
Epoch 7/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.3247 - mae: 0.3247 - val_loss: 0.3080 - val_mae: 0.3080
Epoch 8/100000
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.3094 - mae: 0.3094 - val_loss: 0.3282 - val_mae: 0.3282
Epoch 9/100000
113/113 ━

In [116]:
y_pred = model.predict(X_test)
r2_score(y_test,y_pred)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


0.9287543575512114

In [146]:
#Prediccion de otro valor
df_destruction_pred = pd.read_csv("destruccion_empleo.csv")
df_destruction_pred.drop(columns=["quarter_label", "iso3_code","data_source_notes", "record_id","pct_sector_workforce_displaced"], inplace=True)

In [147]:
df_destruction_pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20800 entries, 0 to 20799
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   country                       20800 non-null  object 
 1   region                        20800 non-null  object 
 2   income_group                  20800 non-null  object 
 3   year                          20800 non-null  int64  
 4   quarter                       20800 non-null  int64  
 5   industry_sector               20800 non-null  object 
 6   sector_automation_risk_score  20800 non-null  float64
 7   gdp_per_capita_usd            20800 non-null  int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 1.3+ MB


In [148]:
df_destruction_pred.head()

,country,region,income_group,year,quarter,industry_sector,sector_automation_risk_score,gdp_per_capita_usd
0,United States,North America,High Income,2020,1,Technology & Software,0.382,63514
1,United States,North America,High Income,2020,1,Finance & Banking,0.608,63514
2,United States,North America,High Income,2020,1,Healthcare & Life Sciences,0.198,63514
3,United States,North America,High Income,2020,1,Manufacturing & Industry,0.720,63514
4,United States,North America,High Income,2020,1,Retail & E-Commerce,0.676,63514


In [149]:
df_destruction_pred["year"][0] = 2027 
df_destruction_pred['quarter'][0] = 4
df_destruction_pred['industry_sector'][0] = 'Technology & Software'

/tmp/ipykernel_20491/1406139506.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_destruction_pred["year"][0] = 2027
/tmp/ipykernel_20491/1406139506.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

In [150]:
df_destruction_pred = pd.get_dummies(df_destruction_pred,dtype=int)

In [151]:
df_destruction_pred.shape

(20800, 110)

In [152]:
X_prediction = scaler.fit_transform(df_destruction_pred)

In [ ]:
#He tratado de limpiarlo lo maximo posible para hacer que la prediccion sea posible, pero no logro obtener un resultado :(
y_pred = model.predict(X_prediction[0])

ValueError: Exception encountered when calling Sequential.call().

[1mCannot take the length of shape with unknown rank.[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=<unknown>, dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>

# EJERCICIO 2

### Prueba 1

In [10]:
folder = listdir('DeFungi')
len(folder)
photos = []
labels = []

In [11]:
for idx, img in enumerate(folder):
    photo = load_img('DeFungi/'+img, color_mode='grayscale',target_size=(40,40))
    photo = img_to_array(photo)
    photos.append(photo)
    img_split = img.split('_')
    img_idx_clean = img_split[0].replace('H','')
    labels.append(float(img_idx_clean))
    


In [12]:
photos = asarray(photos).astype('float32') /255
labels = asarray(labels).astype('float32')

In [13]:
X_train, X_test, y_train, y_test = train_test_split(photos, labels, test_size=0.2, random_state=42)

In [14]:
X_train.shape

(7291, 40, 40, 1)

In [24]:
pd.Series(y_train).unique()

array([2., 1., 5., 3., 6.], dtype=float32)

In [18]:
model = keras.models.Sequential()
model.add(keras.layers.Conv2D(16,(3,3),activation='relu' ,input_shape=X_train.shape[1:]))
model.add(keras.layers.MaxPool2D(2,2))
model.add(keras.layers.Conv2D(32,(3,3), activation='relu'))
model.add(keras.layers.MaxPool2D(2,2))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(200,activation='relu' ,kernel_initializer='he_normal'))
model.add(keras.layers.Dense(40,activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.Dense(5, activation='softmax' ,kernel_initializer='glorot_normal'))

In [19]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
model.compile(loss='categorical_crossentropy', metrics=['accuracy'],optimizer = keras.optimizers.Adam(learning_rate=0.0001,beta_1=0.9,beta_2=0.999))
history = model.fit(X_train, y_train, epochs=100000, validation_split = 0.1, callbacks=[early_stopping_cb])


Epoch 1/100000


ValueError: Arguments `target` and `output` must have the same rank (ndim). Received: target.shape=(None,), output.shape=(None, 5)

In [ ]:
model.evaluate(X_test,y_test)

### Prueba 2

In [4]:
folder = listdir('DeFungi')
len(folder)
photos = []
labels = []

In [ ]:
for idx, img in enumerate(folder):
    photo = load_img('DeFungi/'+img, color_mode='grayscale')
    photo = img_to_array(photo)
    photos.append(photo)
    img_split = img.split('_')
    img_idx_clean = img_split[0].replace('H','')
    img_idx_clean = img_idx_clean.replace({'1': '0', '2': '1', '3': '2', '5': '3', '6': '4'})
    labels.append(float(img_idx_clean))

In [6]:
photos = asarray(photos).astype('float32') /255
labels = asarray(labels).astype('float32')

In [7]:
X_train, X_test, y_train, y_test = train_test_split(photos, labels, test_size=0.2, random_state=42)

In [8]:
model = keras.models.Sequential()
model.add(keras.layers.Conv2D(16,(3,3),activation='relu' ,input_shape=X_train.shape[1:]))
model.add(keras.layers.MaxPool2D(2,2))
model.add(keras.layers.Conv2D(32,(3,3), activation='relu'))
model.add(keras.layers.MaxPool2D(2,2))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(200,activation='relu' ,kernel_initializer='he_normal'))
model.add(keras.layers.Dense(40,activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.Dense(6, activation='softmax' ,kernel_initializer='glorot_normal'))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
W0000 00:00:1778522777.549327   84050 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [9]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
model.compile(loss='categorical_crossentropy', metrics=['accuracy'],optimizer = keras.optimizers.Adam(learning_rate=0.0001,beta_1=0.9,beta_2=0.999))
history = model.fit(X_train, y_train, epochs=100000, validation_split = 0.1, callbacks=[early_stopping_cb])


Epoch 1/100000


ValueError: Arguments `target` and `output` must have the same rank (ndim). Received: target.shape=(None,), output.shape=(None, 6)

- No he logrado entrenar pero imagino que la diferencia entre los 2 casos sea que en una al hacer las imagenes mas pequeñas, habra que usar menos capas de pooling para que no desaparezca la imagen y meter menos en la de salida, y en la otra cogiendo menos imagenes pero con mas dimensión habría que reajustar